# PID evaluation

Two plots from the output of `inference_pid.py`:

1. **Confusion matrix** (needs the true class in the predictions file).
2. **MC-vs-data comparison**, built like Figure 32 in Niels' thesis: stacked MC classes, data points and a Data/MC panel, as a function of the neutrino logit.

**`DEMO = True` runs everything on synthetic data, so the notebook works before the real data is ready.** The fake data is made so that the fit in section 2 should find the scale factors that were put in (noise 2.0, muon 5.0, neutrino 9.0). Set `DEMO = False` and fill in the real file paths when the data is ready.

In [ ]:
DEMO = True   # True: synthetic data. False: use the real files below.

# ---- Real inputs (used when DEMO = False) ----
PRED_FILE = "predictions/pid_predictions_job<ID>.parquet"   # inference_pid.py output for the MC events
PREPARED_DIR = "/groups/icecube/julie/oscNext_prepared/<run>"   # folder written by prepare_pid_data.py
ID_COL = "event_no"      # column in the predictions file that identifies an event
DATA_PRED_FILE = "<inference_pid.py output for the burnsample>"
DATA_TABLE = "<table with event_no and the data weight>"
DATA_WEIGHT_COL = "total_subrun_weight"

# If the MC predictions are only a random SUBSET of the full MC, give the number of events per class
# in the FULL MC (before any cuts), so the weights can be rescaled. None = no rescaling.
N_FULL = None   # e.g. {"noise": ..., "muon": ..., "nue": ..., "numu": ..., "nutau": ...}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iceaggr.models.pid_evaluation import (
    compare_mc_data, mc_class_from_label, neutrino_logit, load_prepared_summary,
    plot_mc_data_comparison, rescale_weights_by_dataset,
)
from iceaggr.models.pid_metrics import CLASS_NAMES, plot_confusion_matrix

MC_WEIGHT_COL = "class_weight"

## Load the tables

Demo: synthetic tables with the same columns as the real ones. Real: the predictions are joined to the truth/weight table on `event_no`; the printout shows how many events were matched, so a mismatch of event numbers shows up here.

In [ ]:
import glob

# --- MC: the test events ---
summary = load_prepared_summary(f"{PREPARED_DIR}/summary.csv")
meta = pd.concat([pd.read_parquet(f, columns=["event_id", "dataset", "label", "pid", MC_WEIGHT_COL])
                  for f in sorted(glob.glob(f"{PREPARED_DIR}/test_meta/*.parquet"))], ignore_index=True)
pred = pd.read_parquet(PRED_FILE)
mc = pred.merge(meta, left_on=ID_COL, right_on="event_id", how="inner")
print(f"MC: {len(pred):,} predictions, {len(mc):,} matched to test_meta ({len(meta):,} test events in total)")

mc["true_class"] = np.minimum(mc["label"], 2)      # 3 classes: the three neutrino flavours become class 2
if "pred_class" not in mc:
    mc["pred_class"] = np.argmax(mc[["pid_noise_pred", "pid_muon_pred", "pid_neutrino_pred"]].to_numpy(), axis=1)

# rescale the weights per dataset, BEFORE any cut
mc["w"], weight_check = rescale_weights_by_dataset(mc[MC_WEIGHT_COL].to_numpy(), mc["dataset"].to_numpy(), summary)
display(weight_check)

# --- Data ---
dpred = pd.read_parquet(DATA_PRED_FILE)
data = dpred.merge(pd.read_parquet(DATA_TABLE, columns=["event_no", DATA_WEIGHT_COL]), on="event_no", how="inner")
print(f"Data: {len(dpred):,} predictions, {len(data):,} matched to the weight table")
print(f"data: {int(data[DATA_WEIGHT_COL].isna().sum()):,} events with a missing weight (they get weight 0)")
data[DATA_WEIGHT_COL] = data[DATA_WEIGHT_COL].fillna(0.0)

## 1. Confusion matrix

Rows are the true class, columns the predicted class. Each row is divided by its total, so a cell is the fraction of that true class that ended up in the column's class.

In [ ]:
n_cls = len(CLASS_NAMES)
cm = np.zeros((n_cls, n_cls), dtype=int)
np.add.at(cm, (mc["true_class"].to_numpy(), mc["pred_class"].to_numpy()), 1)

print(cm)
print(f"accuracy: {np.trace(cm) / cm.sum():.3f}")
plot_confusion_matrix(cm, CLASS_NAMES, path="pid_confusion_matrix.spng")
plt.show()

## 2. MC-vs-data comparison (Figure 32 style)

- The x-axis is the neutrino logit, `log(p / (1 - p))` of `pid_neutrino_pred`.
- The MC classes come from the true `pid`: noise (-1), muon (13), nue (12), numu (14), nutau (16).
- Each MC class is histogrammed with its weight. A Poisson fit then finds one scale factor each for noise, muon and neutrinos (shown as `s=` in the legend), as in the thesis notebook.
- If the MC predictions are only a random subset of the full MC, set `N_FULL` above so that the weights are rescaled per class. **Do not** rescale for physics cuts.
- To make the plot after a cut (for example L3), filter both `mc` and `data` before the next cell.

In [ ]:
mc_logit = neutrino_logit(mc["pid_neutrino_pred"].to_numpy())
data_logit = neutrino_logit(data["pid_neutrino_pred"].to_numpy())
mc_class = mc_class_from_label(mc["label"].to_numpy())
mc_w = mc["w"].to_numpy()

result = compare_mc_data(mc_logit, mc_class, mc_w, data_logit, data[DATA_WEIGHT_COL].to_numpy())

sf = result["scale_factors"]
print(f"scale factors: noise {sf['noise']:.2f}, muon {sf['muon']:.2f}, neutrino {sf['nue']:.2f}")
print(f"MC events: {result['n_mc']:,} | data events: {result['n_data']:,}")

fig, _ = plot_mc_data_comparison(result, title="PID MC-data comparison, test events",
                                 save_path="pid_mc_data_comparison.png")
plt.show()

In [ ]:
# fig, (ax, axr) = plot_mc_data_comparison(result, title="PID MC-data comparison")
# ax.set_xlim(-10, 17)                        # main panel
# ax.legend(fontsize=10, loc="upper left")
# axr.set_ylim(0.5, 1.5)                      # ratio panel
# fig.savefig("my_plot.png", dpi=200, bbox_inches="tight")